In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
import pandas as pd
files = [f'/kaggle/input/datasets/lakshyaupadhyaya/wikipedia-movies-dataset-2016-2026/{year}.csv' for year in range(2016, 2027)]
ds = pd.concat(
    [pd.read_csv(f, index_col=0) for f in files],
    ignore_index=True
)

In [5]:
len(ds)

31060

In [6]:
ds.columns

Index(['title', 'description', 'directed_by', 'written_by', 'produced_by',
       'starring', 'cinematography', 'edited_by', 'release_date', 'country',
       'language'],
      dtype='object')

In [7]:
ds.drop(columns=['written_by', 'cinematography', 'edited_by'], inplace=True)

In [8]:
ds.isnull().sum()

title              2
description      436
directed_by     1272
produced_by     4552
starring        3937
release_date    1182
country         1745
language        2031
dtype: int64

In [9]:
ds.fillna('No data',inplace=True)

In [11]:
ds['release_date'].head(10)

0                                          2016
1                         October 14, 2016(AFF)
2                          August 24, 2016(FFA)
3    March 16, 2016(Cinefilipino Film Festival)
4       November 17, 2016(Cinema One Originals)
5                                  June 3, 2016
6                               22 January 2016
7                                September 2016
8                  24 April 2016(Sci-Fi-London)
9                    15 December 2016(Pakistan)
Name: release_date, dtype: object

In [12]:
def reorder_date(date_str):
    # where date data not present mostly future titles declared
    if pd.isnull(date_str) or date_str == 'No data':
        return 'No date'
    parts=str(date_str).split()
    # if only a year present future movies 2026,2027 etc
    if len(parts)==1 and parts[0].isdigit():
        return f'1 January {parts[0]}'
    # if in month day year format
    if len(parts)==3 and not parts[0].isdigit():
        parts[0],parts[1]=parts[1],parts[0]
        reordered_str=" ".join(parts)
        return reordered_str
    return date_str

def parse_date(date_str):
    if date_str == 'No date':
        return 'No date'
    dt = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
    if pd.notnull(dt):
        return dt.strftime('%d-%m-%Y')
    else:
        return 'No date'


In [14]:
ds['clean_release_date'] = (
    ds['release_date']
    .str.replace(",", " ", regex=False)
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

ds['reordered_date'] = ds['clean_release_date'].apply(reorder_date)
ds['release_date'] = ds['reordered_date'].apply(parse_date)

ds.drop(columns=['clean_release_date', 'reordered_date'], inplace=True)

In [15]:
ds['release_date'].head(10)

0    01-01-2016
1    14-10-2016
2    24-08-2016
3    16-03-2016
4    17-11-2016
5    03-06-2016
6    22-01-2016
7    01-09-2016
8    24-04-2016
9    15-12-2016
Name: release_date, dtype: object

In [16]:
ds.to_csv('movies_cleaned.csv', index=False)